In [1]:
import pandas as pd
import numpy as np

import tensorflow as tf
import networkx as nx

from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

from src.app.mdgcn.transductive.model import SupervisedTransductiveModel
from src.app.mdgcn.inductive.model import SupervisedInductiveModel


In [2]:
# Collecting dataframes from Elliptic datasets.
features_df = pd.read_csv("data/elliptic/elliptic_txs_features.csv", header=None)
classes_df = pd.read_csv("data/elliptic/elliptic_txs_classes.csv")
edges_df = pd.read_csv("data/elliptic/elliptic_txs_edgelist.csv")

# Collecting important features.
transaction_ids = features_df.iloc[:, 0].values # First column of features dataframe is the transaction ID.
time_steps = features_df.iloc[:, 1].values # Second column of features dataframe is the discrete time step.
node_features = features_df.iloc[:, 2:].values # The rest of 165 columns are features related to bitcoin transactions.

### Build the graph

In [3]:
G = nx.DiGraph()

# Every transaction id is a node in my graph.
for transaction in transaction_ids:
    G.add_node(transaction)

# Every transaction operation is an edge connection between two nodes in my graph.
for _, row in edges_df.iterrows():
    G.add_edge(row["txId1"], row["txId2"])

In [4]:
transaction_to_index = {transaction: index for index, transaction in enumerate(transaction_ids)}

In [5]:
N = len(transaction_ids)

# 1. Collect only the indices of the edges (no giant zero matrix)
indices = []
for _, row in edges_df.iterrows():
    i = transaction_to_index[row["txId1"]]
    j = transaction_to_index[row["txId2"]]
    indices.append([i, j])


# 2. Define the values (all 1s in your case) and the shape
# Use float32 to match your original dtype
values = np.ones(len(indices), dtype=np.float32)
dense_shape = [N, N]

# 3. Create the SparseTensor directly
# This uses only a few MBs of RAM instead of 154 GB
adjacent_matrix = tf.sparse.SparseTensor(
    indices=indices,
    values=values,
    dense_shape=dense_shape
)

# 4. (Optional) Reorder indices if they aren't in row-major order
# Many TensorFlow sparse operations require this
adjacent_matrix = tf.sparse.reorder(adjacent_matrix)


In [6]:
def compute_distance_adj_sparse(G, tx_to_idx, K):

    N = len(tx_to_idx)

    # Collect sparse indices per hop
    indices_per_hop = [[] for _ in range(K + 1)]

    for tx in G.nodes():

        src = tx_to_idx[tx]

        lengths = nx.single_source_shortest_path_length(
            G, tx, cutoff=K
        )

        for tgt, d in lengths.items():
            if d <= K:
                tgt_idx = tx_to_idx[tgt]
                indices_per_hop[d].append([src, tgt_idx])

    A_dist_list = []

    for hop in range(K + 1):

        indices = indices_per_hop[hop]

        if len(indices) == 0:
            indices = np.zeros((0, 2), dtype=np.int64)

        values = np.ones(len(indices), dtype=np.float32)

        A_sparse = tf.sparse.SparseTensor(
            indices=indices,
            values=values,
            dense_shape=[N, N]
        )

        A_sparse = tf.sparse.reorder(A_sparse)

        A_dist_list.append(A_sparse)

    return A_dist_list

adjacent_dist_list = compute_distance_adj_sparse(G, transaction_to_index, K=2)

In [7]:
labels_df = classes_df.set_index("txId")

y = []
labels_array = []

for tx in transaction_ids:
    label = labels_df.loc[tx]["class"]

    if label == "unknown":
        y.append(0)
        labels_array.append(0)
    else:
        y.append(1 if label == "1" else 0)
        labels_array.append(1)

y = tf.constant(y, dtype=tf.float32)

# labels: 0 = licit, 1 = illicit
labels = tf.convert_to_tensor(labels_array, dtype=tf.float32)
labels = tf.reshape(labels, (-1, 1))


### Split Data in Train, Validation and Test

In [8]:
indices = np.arange(N)
labels_np = labels.numpy() if isinstance(labels, tf.Tensor) else labels

# Train / temp split
train_idx, temp_idx, y_train, y_temp = train_test_split(
    indices, labels_np, stratify=labels_np, test_size=0.3, random_state=42
)

# Temp -> val / test split
val_idx, test_idx, y_val, y_test = train_test_split(
    temp_idx, labels_np[temp_idx], stratify=labels_np[temp_idx], test_size=0.5, random_state=42
)

# Convert to boolean masks
train_mask = np.zeros(N, dtype=bool)
train_mask[train_idx] = True

val_mask = np.zeros(N, dtype=bool)
val_mask[val_idx] = True

test_mask = np.zeros(N, dtype=bool)
test_mask[test_idx] = True

train_mask = tf.convert_to_tensor(train_mask)
val_mask = tf.convert_to_tensor(val_mask)
test_mask = tf.convert_to_tensor(test_mask)


In [9]:
print('train_mask.shape', train_mask.shape, np.sum(train_mask))
print('val_mask.shape', val_mask.shape, np.sum(val_mask))
print('test_mask.shape', test_mask.shape, np.sum(test_mask))
print('node_features.shape', node_features.shape)

print(tf.reduce_sum(tf.cast(train_mask, tf.int32)))
print(tf.reduce_sum(tf.cast(val_mask, tf.int32)))

multi = tf.cast(train_mask, tf.int32) * tf.cast(val_mask, tf.int32)
print(tf.reduce_sum(multi))


train_mask.shape (203769,) 142638
val_mask.shape (203769,) 30565
test_mask.shape (203769,) 30566
node_features.shape (203769, 165)
tf.Tensor(142638, shape=(), dtype=int32)
tf.Tensor(30565, shape=(), dtype=int32)
tf.Tensor(0, shape=(), dtype=int32)


### Build Transductive Supervised Model

In [10]:
transductive_model = SupervisedTransductiveModel(
    in_dim=node_features.shape[1],
    hidden_dim=64,
    K=2
)

transductive_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)


### Training and Validation Step

In [11]:
transductive_model.fit(
    node_features,
    adjacent_dist_list,
    labels,
    train_mask,
    val_data=(
        node_features,
        adjacent_dist_list,
        labels,
        val_mask
    ),
    epochs=100
)

Epoch 0
Train Loss: 1.598165512084961
Val Loss: 0.7535580992698669
Epoch 10
Train Loss: 1.0273025035858154
Val Loss: 0.5794385671615601
Epoch 20
Train Loss: 0.8202071785926819
Val Loss: 0.4919663369655609
Epoch 30
Train Loss: 0.6812440156936646
Val Loss: 0.44120582938194275
Epoch 40
Train Loss: 0.6078366041183472
Val Loss: 0.4111286997795105
Epoch 50
Train Loss: 0.5496373772621155
Val Loss: 0.3938504159450531
Epoch 60
Train Loss: 0.5091853737831116
Val Loss: 0.3844798803329468
Epoch 70
Train Loss: 0.4841662049293518
Val Loss: 0.37779566645622253
Epoch 80
Train Loss: 0.45758911967277527
Val Loss: 0.37204912304878235
Epoch 90
Train Loss: 0.44090214371681213
Val Loss: 0.36750131845474243


### Evaluation Step

In [12]:
evaluation_results = transductive_model.evaluate_graph(
    node_features,
    adjacent_dist_list,
    labels,
    test_mask
)

pd.DataFrame([evaluation_results]).T.rename(columns={0: 'Score'})

,Score
loss,0.497367
auc,0.862768
precision,0.800995
recall,0.507087
f1,0.621022


### Build Inductive Supervised Model

In [13]:
inductive_model = SupervisedInductiveModel(
    in_dim=node_features.shape[1],
    hidden_dim=64,
    K=2
)

inductive_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

### Training and Validation Step

In [ ]:
inductive_model.fit(
    node_features,
    adjacent_dist_list,
    labels,
    train_mask,
    val_data=(
        node_features,
        adjacent_dist_list,
        labels,
        val_mask
    ),
    epochs=100
)

### Evaluation Step

In [ ]:
inductive_evaluation_results = inductive_model.evaluate_graph(
    node_features,
    adjacent_dist_list,
    labels,
    test_mask
)

pd.DataFrame([inductive_evaluation_results]).T.rename(columns={0: 'Score'})